In [ ]:
# ── Install required packages (run once) ───────────────────────────────
!pip install -U pageindex transformers python-dotenv ipykernel


In [3]:
!pip install jupyter ipywidgets


  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 17.8 MB/s  0:00:00
   ---------------------------------------- 0.0/12.5 MB ? eta -:--:--
   --- ------------------------------------ 1.0/12.5 MB 8.5 MB/s eta 0:00:02
   ----- ---------------------------------- 1.8/12.5 MB 4.8 MB/s eta 0:00:03
   -------- ------------------------------- 2.6/12.5 MB 4.4 MB/s eta 0:00:03
   ---------- ----------------------------- 3.1/12.5 MB 4.1 MB/s eta 0:00:03
   ----------- ---------------------------- 3.7/12.5 MB 3.4 MB/s eta 0:00:03
   ------------ --------------------------- 3.9/12.5 MB 3.3 MB/s eta 0:00:03
   --------------- ------------------------ 4.7/12.5 MB 3.1 MB/s eta 0:00:03
   ----------------- ---------------------- 5.5/12.5 MB 3.2 MB/s eta 0:00:03
   --------------------

In [1]:
# ── Create a .env file (only for PageIndex key) ───────────────────────
env_content = """
PAGEINDEX_API_KEY=2ba8b8123ed744cbb7058013e7609f00
"""
with open(".env", "w") as f:
    f.write(env_content.strip())
print("✅ .env file created")

import os, json, time
from dotenv import load_dotenv

load_dotenv()

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")

print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")

✅ .env file created
PageIndex key loaded: ✅


In [2]:
# ── PageIndex client setup ───────────────────────────────
from pageindex import PageIndexClient
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

print("✅ PageIndex client ready")


✅ PageIndex client ready


In [1]:
# ── Hugging Face model setup (local path, DistilGPT2) ────────────────
from huggingface_hub import snapshot_download
#from transformers import pipeline

# Step 1: Download the model once (if not already cached)
#local_model_path = snapshot_download(
    #repo_id="google/flan-t5-small",   # <-- changed from Mistral-7B to DistilGPT2
    #cache_dir="C:/Users/uloga/hf_models")   # choose your folder


#print("✅ Model downloaded to:", local_model_path)

# Step 2: Load the model from the local folder
#generator = pipeline("text-generation", model=local_model_path)

# Quick test
#output = generator("Hello, how are you?", max_length=50)
#print(output[0]["generated_text"])



from transformers import pipeline

generator = pipeline("text2text-generation", model="google/flan-t5-large")

print(generator("Hello", max_length=50)[0]["generated_text"])



config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

c:\Users\uloga\AppData\Local\Programs\Python\Python39\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\uloga\.cache\huggingface\hub\models--google--flan-t5-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [6]:
from pageindex import PageIndexClient

# Initialize the client with your API key
pi_client = PageIndexClient(api_key="2ba8b8123ed744cbb7058013e7609f00")

# Now you can upload the PDF
PDF_PATH = "./spotify.pdf"
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]


In [11]:
import time
import json



In [12]:
# ── Upload PDF and build tree ─────────────────────────────────────────
PDF_PATH = "./spotify.pdf"

print(f"📤 Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print("✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")

print("⏳ Building tree index for spotify.pdf ...")
while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")

    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed.")
        break

    time.sleep(5)

tree_result = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}\n")
print("🌲 First node preview:")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2)[:1000])

📤 Uploading: ./spotify.pdf
✅ Uploaded!
📋 Document ID: pi-cmue5reh400o60cnvet8xzfmc
⏳ Building tree index for spotify.pdf ...
   Status: processing
   Status: processing
   Status: processing
   Status: completed

✅ Tree index ready!
📊 Top-level sections: 12

🌲 First node preview:
{
  "title": "1. Tech Stack Overview",
  "node_id": "0000",
  "page_index": 1,
  "summary": "This text provides a comprehensive tech stack overview for an application, detailing the specific technologies and tools used across frontend development, backend services, databases and storage, and infrastructure and DevOps.",
  "text": "# 1. Tech Stack Overview\n\nFrontend:\n\n- Next.js (React + TypeScript) for SSR/SSG and SPA behavior.\n- Tailwind CSS with a component library (e.g. Headless UI/Radix UI style).\n- React Query (TanStack Query) for server state.\n- Zustand/Recoil/Jotai for global player state.\n- HLS.js + HTML5 Audio / Web Audio API for playback & visualizations.\n- OAuth2 / OIDC-based auth integratio

In [13]:
# ── Helper functions ─────────────────────────────────────────────────
def print_tree(nodes, indent=0):
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Spotify PDF — Document Structure\n")
print_tree(pageindex_tree)

def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

print(f"🔢 Total nodes: {count_nodes(pageindex_tree)}")
print("   Each node = one retrievable 'chunk' — but semantically meaningful.")

📚 Spotify PDF — Document Structure

[0000] 1. Tech Stack Overview  (p.1)
[0001] 2. Microservices (High-Level)  (p.1)
[0002] 3. Data Storage Layout  (p.3)
[0003] 4. Frontend Structure (Next.js)  (p.3)
[0004] F. Recommendations & Discovery  (p.5)
[0005] G. Social  (p.5)
[0006] H. Subscription / Monetization  (p.5)
[0007] I. Ads (Free Tier)  (p.5)
[0008] J. Creator / Label Portal  (p.5)
[0009] K. Admin / Moderation  (p.5)
[0010] L. Analytics & Reporting  (p.5)
[0011] M. Security & Compliance  (p.5)
🔢 Total nodes: 12
   Each node = one retrievable 'chunk' — but semantically meaningful.


In [14]:
# ── Hugging Face-based tree search ────────────────────────────────────
def llm_tree_search(query: str, tree: list) -> dict:
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150],
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out

    compressed = compress(tree)

    prompt = f"""You are given a query and a document's tree structure.
Identify which node IDs most likely contain the answer.

Query: {query}

Document Tree:
{json.dumps(compressed, indent=2)}

Reply ONLY in this JSON format:
{{
  "thinking": "<your reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = generator(prompt, max_length=300)
    text = response[0]["generated_text"]

    try:
        return json.loads(text)
    except:
        return {"thinking": text, "node_list": []}

In [23]:
from transformers import pipeline

generator = pipeline("text2text-generation", model="google/flan-t5-large")
# Define a helper function
def ask_model(query, max_length=256):
    result = generator(query, max_length=max_length)
    print("🧠 LLM Answer:")
    print(result[0]["generated_text"])


Device set to use cpu


In [24]:
ask_model("Explain Spotify's recommendation system.")

Both `max_new_tokens` (=256) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧠 LLM Answer:
Spotify's recommendation system is based on a user's past usage of the service.
